# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
required={'onnx':'onnx','onnxruntime':'onnxruntime','onnxsim':'onnxsim','torch':'torch','numpy':'numpy'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])


import json, os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 72.2 MB/s eta 0:00:00


In [4]:

import json, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx, onnxruntime as ort
from onnxsim import simplify

TASK_ID='task182'; CH=10; H=W=30
OUT_DIR=Path.cwd()/'generated_models'; OUT_DIR.mkdir(exist_ok=True)
RAW_PATH=OUT_DIR/'task182_raw.onnx'
MODEL_PATH=OUT_DIR/'task182.onnx'

PATTERNS=[
(3,3,((0,1),(1,0),(1,1),(1,2),(2,1))),
(4,1,((0,0),(1,0),(2,0),(3,0))),
(4,4,((0,1),(0,2),(1,0),(1,1),(1,2),(1,3),(2,0),(2,1),(2,2),(2,3),(3,1),(3,2))),
(2,3,((0,0),(0,1),(0,2),(1,0),(1,1),(1,2))),
(1,4,((0,0),(0,1),(0,2),(0,3))),
(4,4,((0,1),(1,1),(2,0),(2,1),(2,2),(2,3),(3,1))),
(3,5,((0,2),(1,1),(1,2),(1,3),(2,0),(2,1),(2,2),(2,3),(2,4))),
(3,3,((0,0),(0,1),(0,2),(1,0),(1,1),(1,2),(2,0),(2,1),(2,2))),
(3,1,((0,0),(1,0),(2,0))),
(3,4,((0,1),(0,2),(1,1),(1,2),(2,0),(2,1),(2,2),(2,3))),
]

class PatternRecolor182Static(nn.Module):
    def __init__(self):
        super().__init__(); self.patterns=[]
        for h,w,coords in PATTERNS:
            idx=len(self.patterns)
            ker=torch.zeros(1,1,h,w)
            for r,c in coords: ker[0,0,r,c]=1.0
            outer=torch.zeros(1,1,h+2,w+2)
            for r,c in coords: outer[0,0,r+1,c+1]=1.0
            bbox=torch.zeros(1,1,h+2,w+2); bbox[:,:,1:h+1,1:w+1]=1.0
            ring=torch.ones(1,1,h+2,w+2); ring[:,:,1:h+1,1:w+1]=0.0
            self.register_buffer(f'ker_{idx}',ker)
            self.register_buffer(f'box_{idx}',torch.ones(1,1,h,w))
            self.register_buffer(f'outer_{idx}',outer)
            self.register_buffer(f'bbox_{idx}',bbox)
            self.register_buffer(f'ring_{idx}',ring)
            self.patterns.append((h,w,coords,len(coords)))
    def pattern_pixels(self,m,idx):
        h,w,coords,area=self.patterns[idx]
        ker=getattr(self,f'ker_{idx}'); box=getattr(self,f'box_{idx}')
        local=(F.conv2d(m,ker)==float(area)).float()*(F.conv2d(m,box)==float(area)).float()
        mp=F.pad(m,(1,1,1,1))
        outer=getattr(self,f'outer_{idx}'); bbox=getattr(self,f'bbox_{idx}'); ring=getattr(self,f'ring_{idx}')
        bounded=(F.conv2d(mp,outer)==float(area)).float()*(F.conv2d(mp,bbox)==float(area)).float()*(F.conv2d(mp,ring)==0.0).float()
        det=local*bounded
        return F.conv_transpose2d(det,ker).clamp(0,1)
    def forward(self,x):
        active=(x.sum(1,keepdim=True)>0).float(); blue=x[:,1:2]; gray=x[:,5:6]
        gleft=(torch.cumsum(gray,dim=3)>0).float(); gright=torch.flip((torch.cumsum(torch.flip(gray,[3]),dim=3)>0).float(),[3])
        gup=(torch.cumsum(gray,dim=2)>0).float(); gdown=torch.flip((torch.cumsum(torch.flip(gray,[2]),dim=2)>0).float(),[2])
        inside=gleft*gright*gup*gdown
        zero=blue*0; add=[zero for _ in range(CH)]; recolor=zero
        for idx in range(len(self.patterns)):
            bpix=self.pattern_pixels(blue,idx)
            for k in range(2,CH):
                if k==5: continue
                tpix=self.pattern_pixels(x[:,k:k+1]*inside,idx)
                has=(tpix.sum((2,3),keepdim=True)>0).float()
                cand=bpix*has
                add[k]=(add[k]+cand).clamp(0,1); recolor=(recolor+cand).clamp(0,1)
        out=[x[:,k:k+1].clone() for k in range(CH)]
        out[1]=blue*(1-recolor)
        for k in range(2,CH): out[k]=(out[k]+add[k]).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1); out[0]=(1-occ)*active
        return torch.cat(out,1)

def load_task(task_id=TASK_ID):
    for root in [Path.cwd(), Path.cwd().parent, Path('/mnt/data'),Path(COMPETITION)]:
        p=root/f'{task_id}.json'
        if p.exists(): return json.load(open(p)), p
    raise FileNotFoundError(task_id)

def onehot(grid):
    a=np.array(grid); h,w=a.shape
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    for k in range(CH): x[0,k,:h,:w]=(a==k)
    return x,h,w


In [5]:
task, task_path = load_task()
print('task path:', task_path)
print({k: len(task.get(k, [])) for k in ['train','test','arc-gen']})

task path: /kaggle/input/competitions/neurogolf-2026/task182.json
{'train': 4, 'test': 1, 'arc-gen': 262}


In [6]:
model=PatternRecolor182Static().eval()
torch.onnx.export(model, torch.zeros(1,CH,H,W), str(RAW_PATH), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)
raw=onnx.load(str(RAW_PATH))
simplified, check = simplify(raw)
assert check
onnx.save(simplified, str(MODEL_PATH))
print('saved', MODEL_PATH, 'size', MODEL_PATH.stat().st_size)

/tmp/ipykernel_16/1275473197.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.zeros(1,CH,H,W), str(RAW_PATH), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:305: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/usr/local/lib/python3.12/dist-packages/torch

saved /kaggle/working/generated_models/task182.onnx size 288746


In [7]:
m=onnx.load(str(MODEL_PATH))
onnx.checker.check_model(m)
ops={}
for node in m.graph.node: ops[node.op_type]=ops.get(node.op_type,0)+1
shape_in=[d.dim_value or d.dim_param for d in m.graph.input[0].type.tensor_type.shape.dim]
shape_out=[d.dim_value or d.dim_param for d in m.graph.output[0].type.tensor_type.shape.dim]
forbidden=[op for op in ['Loop','Scan','NonZero','Unique','Script','Function'] if ops.get(op,0)]
risk=[op for op in ['Shape','Gather','ScatterND','Range','Expand','ConstantOfShape'] if ops.get(op,0)]
print('input:', shape_in, 'output:', shape_out)
print('size:', MODEL_PATH.stat().st_size)
print('forbidden:', forbidden, 'risk:', risk)
assert shape_in == [1,10,30,30] and shape_out == [1,10,30,30]
assert MODEL_PATH.stat().st_size < 1_400_000
assert not forbidden and not risk

input: [1, 10, 30, 30] output: [1, 10, 30, 30]
size: 288746
forbidden: [] risk: []


In [8]:
sess=ort.InferenceSession(str(MODEL_PATH), providers=['CPUExecutionProvider'])
def validate_section(sec):
    exact=0
    for ex in task[sec]:
        x,h,w=onehot(ex['input'])
        pred=sess.run(None, {'input': x})[0].argmax(1)[0,:h,:w]
        exact += int(np.array_equal(pred, np.array(ex['output'])))
    return {'exact': exact, 'total': len(task[sec])}
validation={sec: validate_section(sec) for sec in ['train','test','arc-gen']}
print(validation)
assert validation['train']['exact']==validation['train']['total']
assert validation['test']['exact']==validation['test']['total']
assert validation['arc-gen']['exact']==validation['arc-gen']['total']
idx=list(range(len(task['arc-gen']))); random.Random(0).shuffle(idx); hold=set(idx[:max(1,int(round(.3*len(idx))))])
hold_exact=0
for i in hold:
    ex=task['arc-gen'][i]; x,h,w=onehot(ex['input']); pred=sess.run(None, {'input': x})[0].argmax(1)[0,:h,:w]
    hold_exact += int(np.array_equal(pred, np.array(ex['output'])))
print('30% arc-gen holdout:', hold_exact, '/', len(hold))
assert hold_exact==len(hold)

{'train': {'exact': 4, 'total': 4}, 'test': {'exact': 1, 'total': 1}, 'arc-gen': {'exact': 262, 'total': 262}}
30% arc-gen holdout: 79 / 79


In [9]:
with zipfile.ZipFile(Path.cwd()/'submission.zip','w',zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_PATH, 'task182.onnx')
print('wrote', Path.cwd()/'submission.zip')

wrote /kaggle/working/submission.zip
